# 04b · Ablation — Start-rescue & Terminal Extrapolation

**Purpose.** Measure how much each rescue component adds. PRRSV **start-rescue** and FMDV **terminal extrapolation** (the latter's output is currently MISSING and must be regenerated). Feeds **Figure 5**.

## Inputs

PRRS + FMD ref/query. Toggle each rescue component on/off and re-score.

In [ ]:
from pathlib import Path
import sys, time

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA   = ROOT / "app" / "data"
CONFIG = ROOT / "app" / "config"

# References (verify these are the intended ref records for the paper):
FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"        # alt: FMD_FJ175661_Anno.gb
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"       # alt: PRRS_MT746146_Anno.gb
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REFS   = {"ref_1": DATA / "PED" / "PED_ref_1.gb",
              "ref_2": DATA / "PED" / "PED_ref_2.gb"}
PED_QUERY  = DATA / "PED" / "PED_100seqs.gb"

# Run toggle: keep False for a fast smoke test, True for the full 100-record run.
RUN_FULL = False
SAMPLE_N = 10


In [ ]:
# outputs land inside this unit folder so figures/tables sit next to the notebook
UNIT_DIR = ROOT / "app" / "validation" / "07_ablation_runtime"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## Run — baseline vs rescue-on

> ⚠️ **TODO**: `run_tblastn_against_truth` runs the full pipeline (rescue ON). For the ablation you need a rescue-OFF pass. Options: (a) add a `rescue=False` kwarg path through `run_tblastn_batch` / `process_one_query_record`, or (b) call the lifter directly and skip `extrapolate_terminal_boundaries` / start-rescue scoring. Wire the toggle below.

In [ ]:
from app.validation._shared.validation_utils import run_tblastn_against_truth, summarize_comparison

# rescue ON (current default)
prrs_on, _  = run_tblastn_against_truth("prrs", PRRS_REF, PRRS_QUERY, OUT / "ablation/prrs_on", progress=True)
fmd_on,  _  = run_tblastn_against_truth("fmd",  FMD_REF,  FMD_QUERY,  OUT / "ablation/fmd_on",  progress=True)

# rescue OFF  -- TODO: implement the OFF path, then score identically
prrs_off = None   # start-rescue disabled
fmd_off  = None   # terminal-extrapolation disabled

## Metrics — delta accuracy from each component

In [ ]:
def tag(df, exp):
    df = df.copy(); df["experiment"] = exp; return df

def compare(on, off, virus):
    if off is None:
        print(f"[skip] {virus}: OFF pass not wired yet"); return None
    both = pd.concat([tag(on, "rescue_on"), tag(off, "rescue_off")], ignore_index=True)
    return summarize_comparison(both, ["experiment"])

prrs_cmp = compare(prrs_on, prrs_off, "prrs")   # start-rescue effect
fmd_cmp  = compare(fmd_on,  fmd_off,  "fmd")    # terminal-extrapolation effect
prrs_cmp

## Figure

> ⚠️ **TODO**: bar chart exact% rescue_off vs rescue_on cho PRRSV ORF7/ORF1b và FMDV mat_peptide termini; save `prrsv_start_rescue_exact_comparison.png`, `fmd_terminal_extrapolation_comparison.png`.

## Interpretation

> ⚠️ **TODO**: mỗi thành phần rescue đóng góp bao nhiêu điểm accuracy; component nào đáng giữ.